# 1. Notebook Overview

## Objective

This notebook implements the first Extract, Transform and Load (ETL) process of the Oil & Gas Data Warehouse project.

The objective is to populate all dimension tables previously designed and created in the dimensional model using the original production and hydraulic fracturing datasets.

The ETL process includes the extraction of source data, the application of the required transformations, and the loading of cleaned and standardized records into the corresponding dimension tables in MySQL.

At the end of this notebook, the dimensional model will contain all descriptive entities required for loading the fact tables in the next stage of the project

## Dimensions Loaded

The following dimension tables are populated in this notebook:

|Dimension|Description|
|---------|-----------|
|dim_time|Calendar dimension used by all business events.|
|dim_well|Stores the static characteristics of each well.|
|dim_geology|Stores geological information associated with each well.|
|dim_field_area|Stores concession areas and producing field information.|
|dim_location|Stores the geographical location of each well.|
|dim_company|Stores the operating companies.|

## ETL Workflow

The ETL process implemented in this notebook follows the workflow below:

```text
Source Data
      │
      ▼
Data Extraction
      │
      ▼
Data Cleaning
      │
      ▼
Data Transformation
      │
      ▼
Dimension Loading
      │
      ▼
Validation
```

## Expected Outcome

Upon completion of this notebook:

- All dimension tables will be populated.
- Duplicate records will be removed.
- Business keys will be preserved.
- Surrogate keys will be automatically generated by MySQL.
- The Data Warehouse will be ready for loading the production and hydraulic fracturing fact tables in the following notebook.

# 2. Prerequisites

## Objective

Prepare the working environment required to execute the ETL process for loading the dimension tables into the Oil & Gas Data Warehouse.

## Required Libraries

Import the Python libraries used throughout the notebook.

In [1]:
import pandas as pd
import mysql.connector

## Database Connection

Create the connection to the MySQL server and select the oil_gas_dw database.

In [2]:
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="123456789"
)

cursor = conn.cursor(buffered=True)

cursor.execute("USE oil_gas_dw")
print("Database selected.")

Database selected.


## Helper Functions

The following helper functions are defined to simplify database interaction throughout the ETL process. They encapsulate the most common SQL operations, improve code readability, and promote code reuse across the notebook.

|Function|Description|
|--------|-----------|
|execute_sql()|Executes SQL statements that modify the database (CREATE, INSERT, UPDATE, DELETE, etc.) and commits the transaction.|
|execute_query()|Executes SQL queries and returns all retrieved records.|
|describe_table()|Returns the structure of a table as a Pandas DataFrame using the SQL DESCRIBE command.|
|show_create_table()|Returns the complete SQL definition used to create a table.|
|get_row_count()|Returns the total number of records stored in a table.|
|load_to_table()|Loads multiple records into a MySQL table using executemany() for efficient batch insertion.|

In [3]:
# Execute SQL statements that modify the database.

def execute_sql(sql, message=None):
    try:
        cursor.execute(sql)
        conn.commit()
        if message:
            print(message)
    except mysql.connector.Error as err:
        print(f"Error: {err}")

In [4]:
# Execute a SQL query and return all results.

def execute_query(sql):
    cursor.execute(sql)
    return cursor.fetchall()

In [5]:
# Display the structure of a database table.

def describe_table(table_name):
    return pd.DataFrame(
        execute_query(f"DESCRIBE {table_name}"),
        columns=[
            "Field",
            "Type",
            "Null",
            "Key",
            "Default",
            "Extra"
        ]
    )

In [6]:
# Return the CREATE TABLE statement.

def show_create_table(table_name):
    return execute_query(f"SHOW CREATE TABLE {table_name}")[0][1]

In [8]:
# Return the number of records in a table.

def get_row_count(table_name):
    return execute_query(
        f"SELECT COUNT(*) FROM {table_name}"
    )[0][0]

In [9]:
# Helper Functions for MySQL ETL Loading

def load_to_table(insert_sql, dataframe, table_name):
    try:

        records = [
            tuple(
                None if pd.isna(value) else value
                for value in row
            )
            for row in dataframe.itertuples(index=False, name=None)
        ]

        cursor.executemany(insert_sql, records)
        conn.commit()

        print(f"✓ {cursor.rowcount} records inserted into '{table_name}'.")

    except mysql.connector.Error as err:
        conn.rollback()
        print(f"Error loading '{table_name}': {err}")

## Load Source Datasets
Load the original datasets used throughout the ETL process.

In [10]:
# Load the complete production dataset.
production_df = pd.read_csv(
    "../data/raw/produccin-de-pozos-de-gas-y-petrleo-no-convencional.csv", 
    low_memory=False
)

In [11]:
fracture_path = "../data/raw/datos-de-fractura-de-pozos-de-hidrocarburos-adjunto-iv-actualizacin-diaria.csv"
fracture_df = pd.read_csv(fracture_path)

## Validation
Verify that the environment has been correctly configured before starting the ETL process.

In [49]:
print(production_df.shape)
print(fracture_df.shape)
cursor.execute("SELECT DATABASE();")
print(cursor.fetchone())

(415903, 40)
(4806, 30)
('oil_gas_dw',)


### Prerequisites Validation

The validation confirms that the working environment has been successfully configured.

Both source datasets were loaded correctly, the connection to the MySQL server was established, and the oil_gas_dw database was successfully selected. Therefore, the environment is ready to execute the ETL process for loading the dimension tables.

# 3. ETL Workflow

## Objective

This section describes the general Extract, Transform and Load (ETL) workflow implemented throughout this notebook.

The objective is to populate each dimension table of the Oil & Gas Data Warehouse by extracting data from the source datasets, applying the required transformations, and loading the cleaned information into MySQL while preserving data consistency and integrity.

## ETL Pipeline

The ETL process follows the workflow illustrated below.

```text
Source Dataset
       │
       ▼
Column Selection
       │
       ▼
Data Cleaning
       │
       ▼
Data Transformation
       │
       ▼
Duplicate Removal
       │
       ▼
Dimension Loading
       │
       ▼
Validation
```

## General Workflow Description

Each dimension is populated following the same standardized ETL procedure.

1. Extract
- Read the required columns from the source dataset.
2. Transform
- Select only the attributes required by the dimensional model.
- Remove duplicate records.
- Handle missing values when necessary.
- Standardize data types and formats.
3. Load
- Insert the transformed records into the corresponding dimension table.
4. Validate
- Verify that the records were successfully inserted.
- Compare the number of loaded records with the transformed dataset.
- Inspect the resulting table structure and contents when appropriate.

## Dimension Loading Sequence

The dimension tables will be populated in the following order:

|Step|Dimension|
|----|---------|
|1|dim_time|
|2|dim_well|
|3|dim_geology|
|4|dim_field_area|
|5|dim_location|
|6|dim_company|

This sequence ensures that all descriptive entities are available before loading the fact tables in the next stage of the project.

## Expected Outcome

Upon completion of this workflow:

- All dimension tables will be populated.
- Business keys from the source datasets will be preserved.
- Surrogate keys will be automatically generated by MySQL.
- The dimensional model will be ready for loading the production and hydraulic fracturing fact tables.

# 4. Dimension Loading

## Objective

The objective of this section is to populate all dimension tables defined in the dimensional model.

Each dimension is loaded independently following a standardized ETL process that extracts data from the source datasets, applies the required transformations, and inserts the resulting records into the corresponding MySQL dimension table.

All dimension tables are populated before loading the fact tables in order to ensure referential integrity through surrogate keys.

## ETL Methodology

Each dimension follows the same ETL methodology throughout this notebook.

```text
Objective

↓

Source Data

↓

Transformation

↓

Load

↓

Validation
```

This standardized workflow guarantees consistency, reproducibility, and readability across all ETL processes implemented in this project.

## Dimensions Included

The following dimension tables will be loaded:

|Section|Dimension|Source Dataset|
|-------|---------|--------------|
|4.1|dim_time|production_df|
|4.2|dim_well|production_df + fracture_df|
|4.3|dim_geology|production_df + fracture_df|
|4.4|dim_field_area|production_df + fracture_df|
|4.5|dim_location|production_df|
|4.6|dim_company|production_df|

## Expected Outcome

Upon completion of this section, all dimension tables of the Oil & Gas Data Warehouse will be populated with clean, standardized, and validated data, providing the descriptive entities required for loading the fact tables.

## 4.1 Load dim_time

### Objective

Populate the dim_time table with a complete calendar dimension derived from the production dates available in the source dataset.

This dimension provides the temporal attributes required for analytical reporting and serves as the primary date reference for both fact tables in the Data Warehouse.

### Source Data

#### Source dataset
```python
production_df
```

#### Source column

|Column|Description|
|------|-----------|
|fecha_data|Monthly production date used to generate the calendar dimension.|

The fecha_data column contains the reporting date for each production record and serves as the source for generating all calendar attributes.

### Transformation

The following transformations will be applied:

1. Extract the unique values from fecha_data.
2. Convert the values to the datetime data type.
3. Sort the dates in ascending order.
4. Generate the calendar attributes required by dim_time:
- full_date
- year
- quarter
- month
- month_name
- day
- day_of_week
- day_name
- week_of_year
5. Build the final DataFrame following the dimensional model definition.

### Load

Insert the transformed records into the dim_time table.
The surrogate key (time_key) will be automatically generated by MySQL using the AUTO_INCREMENT property.

### Validation

After loading the data:

- Verify the total number of inserted records.
- Inspect the table structure.
- Confirm that one row exists for each unique reporting date.
- Verify that the generated calendar attributes are correct.

### Implementation

In [50]:
# =======================================================================
# 1. Extract
# =======================================================================

production_dates = production_df[["fecha_data"]].copy()
production_dates.head()

,fecha_data
0,2015-01-31
1,2017-01-31
2,2015-01-31
3,2018-01-31
4,2015-01-31


In [51]:
# =======================================================================
# 2. Transform
# =======================================================================

production_dates["fecha_data"] = pd.to_datetime(
    production_dates["fecha_data"]
)

production_dates = (
    production_dates
    .drop_duplicates()
    .sort_values("fecha_data")
    . reset_index(drop=True)
                    
)

dim_time_df = pd.DataFrame()

dim_time_df["full_date"] = production_dates["fecha_data"]
dim_time_df["year"] = dim_time_df["full_date"].dt.year
dim_time_df["quarter"] = dim_time_df["full_date"].dt.quarter
dim_time_df["month"] = dim_time_df["full_date"].dt.month
dim_time_df["month_name"] = dim_time_df["full_date"].dt.month_name()
dim_time_df["day"] = dim_time_df["full_date"].dt.day
dim_time_df["day_of_week"] = (dim_time_df["full_date"].dt.isocalendar().day)
dim_time_df["day_name"] = dim_time_df["full_date"].dt.day_name()
dim_time_df["week_of_year"] = (dim_time_df["full_date"].dt.isocalendar().week)
dim_time_df.head()

,full_date,year,quarter,month,month_name,day,day_of_week,day_name,week_of_year
0,2006-01-31,2006,1,1,January,31,2,Tuesday,5
1,2006-02-28,2006,1,2,February,28,2,Tuesday,9
2,2006-03-31,2006,1,3,March,31,5,Friday,13
3,2006-04-30,2006,2,4,April,30,7,Sunday,17
4,2006-05-31,2006,2,5,May,31,3,Wednesday,22


In [52]:
dim_time_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 246 entries, 0 to 245
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   full_date     246 non-null    datetime64[ns]
 1   year          246 non-null    int32         
 2   quarter       246 non-null    int32         
 3   month         246 non-null    int32         
 4   month_name    246 non-null    object        
 5   day           246 non-null    int32         
 6   day_of_week   246 non-null    UInt32        
 7   day_name      246 non-null    object        
 8   week_of_year  246 non-null    UInt32        
dtypes: UInt32(2), datetime64[ns](1), int32(4), object(2)
memory usage: 12.1+ KB


In [53]:
dim_time_df.describe(include="all")

,full_date,year,quarter,month,month_name,day,day_of_week,day_name,week_of_year
count,246,246.000000,246.000000,246.000000,246,246.000000,246.0,246,246.0
unique,NaN,NaN,NaN,NaN,12,NaN,<NA>,7,<NA>
top,NaN,NaN,NaN,NaN,January,NaN,<NA>,Tuesday,<NA>
freq,NaN,NaN,NaN,NaN,21,NaN,<NA>,37,<NA>
mean,2016-04-15 04:46:49.756097536,2015.756098,2.475610,6.426829,NaN,30.430894,4.004065,NaN,26.365854
min,2006-01-31 00:00:00,2006.000000,1.000000,1.000000,NaN,28.000000,1.0,NaN,1.0
25%,2011-03-07 18:00:00,2011.000000,1.000000,3.000000,NaN,30.000000,2.0,NaN,13.0
50%,2016-04-15 00:00:00,2016.000000,2.000000,6.000000,NaN,31.000000,4.0,NaN,26.0
75%,2021-05-23 06:00:00,2021.000000,3.000000,9.000000,NaN,31.000000,6.0,NaN,39.0
max,2026-06-30 00:00:00,2026.000000,4.000000,12.000000,NaN,31.000000,7.0,NaN,53.0


In [54]:
dim_time_df = dim_time_df.astype({
    "year": "int16",
    "quarter": "int8",
    "month": "int8",
    "day": "int8",
    "day_of_week": "int8",
    "week_of_year": "int8"
})

In [55]:
dim_time_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 246 entries, 0 to 245
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   full_date     246 non-null    datetime64[ns]
 1   year          246 non-null    int16         
 2   quarter       246 non-null    int8          
 3   month         246 non-null    int8          
 4   month_name    246 non-null    object        
 5   day           246 non-null    int8          
 6   day_of_week   246 non-null    int8          
 7   day_name      246 non-null    object        
 8   week_of_year  246 non-null    int8          
dtypes: datetime64[ns](1), int16(1), int8(5), object(2)
memory usage: 7.6+ KB


In [56]:
# =======================================================================
# 3. Load
# =======================================================================

insert_sql = """
INSERT INTO dim_time (
    full_date,
    year,
    quarter,
    month,
    month_name,
    day,
    day_of_week,
    day_name,
    week_of_year
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

load_to_table(insert_sql, dim_time_df, "dim_time")

Error loading 'dim_time': 1062 (23000): Duplicate entry '2006-01-31' for key 'dim_time.full_date'


In [57]:
# =======================================================================
# 4. Validation
# =======================================================================

# Verify total number of records
print(f"Rows in dim_time: {get_row_count('dim_time')}")

Rows in dim_time: 246


In [58]:
# Verify table structure
describe_table("dim_time")

,Field,Type,Null,Key,Default,Extra
0,time_key,int,NO,PRI,None,auto_increment
1,full_date,date,NO,UNI,None,
2,year,smallint,NO,,None,
3,quarter,tinyint,NO,,None,
4,month,tinyint,NO,,None,
5,month_name,varchar(15),NO,,None,
6,day,tinyint,NO,,None,
7,day_of_week,tinyint,NO,,None,
8,day_name,varchar(15),NO,,None,
9,week_of_year,tinyint,NO,,None,


In [59]:
# Verify SQL definition

print(show_create_table("dim_time"))

CREATE TABLE `dim_time` (
  `time_key` int NOT NULL AUTO_INCREMENT,
  `full_date` date NOT NULL,
  `year` smallint NOT NULL,
  `quarter` tinyint NOT NULL,
  `month` tinyint NOT NULL,
  `month_name` varchar(15) COLLATE utf8mb4_unicode_ci NOT NULL,
  `day` tinyint NOT NULL,
  `day_of_week` tinyint NOT NULL,
  `day_name` varchar(15) COLLATE utf8mb4_unicode_ci NOT NULL,
  `week_of_year` tinyint NOT NULL,
  PRIMARY KEY (`time_key`),
  UNIQUE KEY `full_date` (`full_date`)
) ENGINE=InnoDB AUTO_INCREMENT=739 DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci


In [60]:
# Inspect loaded records

pd.DataFrame(
    execute_query(
        """
        SELECT *
        FROM dim_time
        LIMIT 10
        """
    ),
    columns=[
        "time_key",
        "full_date",
        "year",
        "quarter",
        "month",
        "month_name",
        "day",
        "day_of_week",
        "day_name",
        "week_of_year"
    ]
)

,time_key,full_date,year,quarter,month,month_name,day,day_of_week,day_name,week_of_year
0,1,2006-01-31,2006,1,1,January,31,2,Tuesday,5
1,2,2006-02-28,2006,1,2,February,28,2,Tuesday,9
2,3,2006-03-31,2006,1,3,March,31,5,Friday,13
3,4,2006-04-30,2006,2,4,April,30,7,Sunday,17
4,5,2006-05-31,2006,2,5,May,31,3,Wednesday,22
5,6,2006-06-30,2006,2,6,June,30,5,Friday,26
6,7,2006-07-31,2006,3,7,July,31,1,Monday,31
7,8,2006-08-31,2006,3,8,August,31,4,Thursday,35
8,9,2006-09-30,2006,3,9,September,30,6,Saturday,39
9,10,2006-10-31,2006,4,10,October,31,2,Tuesday,44


### Conclusion

The dim_time dimension was successfully populated from the production reporting dates. All calendar attributes were generated correctly, surrogate keys were automatically assigned by MySQL, and the validation process confirmed the integrity of the loaded records.

## 4.2 Load dim_well

### Objective

Populate the dim_well dimension using the well-level information available in the production and fracturing datasets.

Each record represents a unique well and contains stable descriptive attributes that characterize the well independently of production or fracturing events.

### Source Data
#### Source dataset
```python
production_df
fracture_df
```
#### Source column

|Dataset|Columns|
|-------|-------|
|production_df|idpozo, sigla, profundidad, proyecto, clasificacion, subclasificacion|
|fracture_df|idpozo, longitud_rama_horizontal_m|

The production dataset provides the main descriptive information for each well, while the fracturing dataset contributes the horizontal branch length (longitud_rama_horizontal_m).

### Transformation

The following transformations will be applied:

1. Extract the well descriptive attributes from both datasets.
2. Keep one record per idpozo.
3. Merge the production and fracturing information.
4. Preserve all wells appearing in the production dataset.
5. Build the final DataFrame following the dimensional model definition.

### Load

Insert the transformed records into the dim_well table.

The surrogate key (well_key) will be automatically generated by MySQL using the AUTO_INCREMENT property.

### Validation

After loading the data:

- Verify the total number of inserted records.
- Verify that one record exists per idpozo.
- Inspect the table structure.
- Verify that the surrogate key was generated correctly.
- Inspect a sample of loaded records.

### Implementation

In [61]:
# =======================================================================
# 1. Extract
# =======================================================================

production_wells = production_df[
    [
        "idpozo",
        "sigla",
        "profundidad",
        "proyecto",
        "clasificacion",
        "subclasificacion"
    ]
].copy()

fracture_wells = fracture_df[
    [
        "idpozo",
        "longitud_rama_horizontal_m"
    ]
].copy()

production_wells.head()

,idpozo,sigla,profundidad,proyecto,clasificacion,subclasificacion
0,132772,APA.Nq.Gu-1202(d),3703.0,GAS PLUS,EXPLOTACION,DESARROLLO
1,130300,AEA.NQ.RCo-1024,2633.0,GAS PLUS,EXPLOTACION,DESARROLLO
2,130518,AEA.NQ.RCo-1023,2638.0,GAS PLUS,EXPLOTACION,DESARROLLO
3,133429,AEA.NQ.Gu-1190,2758.0,GAS PLUS,EXPLOTACION,DESARROLLO
4,143846,APA.RN.EFO-144(d),3949.0,GAS PLUS,EXPLOTACION,DESARROLLO


In [62]:
fracture_wells.head()

,idpozo,longitud_rama_horizontal_m
0,159910,0.0
1,159910,0.0
2,159219,1437.3
3,159220,1518.3
4,159221,1482.3


In [63]:
# =======================================================================
# 2. Transform
# =======================================================================

production_wells = (
    production_wells
    .drop_duplicates(subset="idpozo")
    .reset_index(drop=True)
)

In [64]:
fracture_wells = (
    fracture_wells
    .drop_duplicates(subset="idpozo")
    .reset_index(drop=True)
)

In [65]:
print(production_wells.shape)
print(fracture_wells.shape)

(5041, 6)
(4562, 2)


In [66]:
# Merge production and fracturing information

dim_well_df = production_wells.merge(
    fracture_wells,
    on="idpozo",
    how="left"
)

In [67]:
dim_well_df.head()

,idpozo,sigla,profundidad,proyecto,clasificacion,subclasificacion,longitud_rama_horizontal_m
0,132772,APA.Nq.Gu-1202(d),3703.0,GAS PLUS,EXPLOTACION,DESARROLLO,NaN
1,130300,AEA.NQ.RCo-1024,2633.0,GAS PLUS,EXPLOTACION,DESARROLLO,NaN
2,130518,AEA.NQ.RCo-1023,2638.0,GAS PLUS,EXPLOTACION,DESARROLLO,87.0
3,133429,AEA.NQ.Gu-1190,2758.0,GAS PLUS,EXPLOTACION,DESARROLLO,NaN
4,143846,APA.RN.EFO-144(d),3949.0,GAS PLUS,EXPLOTACION,DESARROLLO,NaN


In [68]:
dim_well_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5041 entries, 0 to 5040
Data columns (total 7 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   idpozo                      5041 non-null   int64  
 1   sigla                       5041 non-null   object 
 2   profundidad                 5041 non-null   float64
 3   proyecto                    5041 non-null   object 
 4   clasificacion               5037 non-null   object 
 5   subclasificacion            5037 non-null   object 
 6   longitud_rama_horizontal_m  3765 non-null   float64
dtypes: float64(2), int64(1), object(4)
memory usage: 275.8+ KB


In [69]:
print(dim_well_df.shape)
print(dim_well_df["idpozo"].duplicated().sum())
print(dim_well_df["idpozo"].nunique())

(5041, 7)
0
5041


In [70]:
# =======================================================================
# 3. Load
# =======================================================================

insert_sql = """
INSERT INTO dim_well (
    idpozo,
    sigla,
    profundidad,
    proyecto,
    clasificacion,
    subclasificacion,
    longitud_rama_horizontal_m
)
VALUES (%s, %s, %s, %s, %s, %s, %s)
"""

load_to_table(insert_sql, dim_well_df, "dim_well")

Error loading 'dim_well': 1062 (23000): Duplicate entry '132772' for key 'dim_well.idpozo'


In [71]:
# =======================================================================
# 4. Validation
# =======================================================================

# Verify total number of records

print(f"Rows in dim_well: {get_row_count('dim_well')}")

Rows in dim_well: 5041


In [72]:
# Verify table structure

describe_table("dim_well")

,Field,Type,Null,Key,Default,Extra
0,well_key,int,NO,PRI,None,auto_increment
1,idpozo,int,NO,UNI,None,
2,sigla,varchar(50),NO,,None,
3,profundidad,"decimal(10,2)",YES,,None,
4,longitud_rama_horizontal_m,"decimal(10,2)",YES,,None,
5,proyecto,varchar(100),YES,,None,
6,clasificacion,varchar(100),YES,,None,
7,subclasificacion,varchar(100),YES,,None,


In [73]:
# Verify SQL definition

print(show_create_table("dim_well"))

CREATE TABLE `dim_well` (
  `well_key` int NOT NULL AUTO_INCREMENT,
  `idpozo` int NOT NULL,
  `sigla` varchar(50) COLLATE utf8mb4_unicode_ci NOT NULL,
  `profundidad` decimal(10,2) DEFAULT NULL,
  `longitud_rama_horizontal_m` decimal(10,2) DEFAULT NULL,
  `proyecto` varchar(100) COLLATE utf8mb4_unicode_ci DEFAULT NULL,
  `clasificacion` varchar(100) COLLATE utf8mb4_unicode_ci DEFAULT NULL,
  `subclasificacion` varchar(100) COLLATE utf8mb4_unicode_ci DEFAULT NULL,
  PRIMARY KEY (`well_key`),
  UNIQUE KEY `idpozo` (`idpozo`)
) ENGINE=InnoDB AUTO_INCREMENT=15124 DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci


In [74]:
# Inspect loaded records

pd.DataFrame(
    execute_query(
        """
        SELECT *
        FROM dim_well
        LIMIT 10
        """
    ),
    columns=[
        "well_key",
        "idpozo",
        "sigla",
        "profundidad",
        "longitud_rama_horizontal_m",
        "proyecto",
        "clasificacion",
        "subclasificacion"
    ]
)

,well_key,idpozo,sigla,profundidad,longitud_rama_horizontal_m,proyecto,clasificacion,subclasificacion
0,1,132772,APA.Nq.Gu-1202(d),3703.00,None,GAS PLUS,EXPLOTACION,DESARROLLO
1,2,130300,AEA.NQ.RCo-1024,2633.00,None,GAS PLUS,EXPLOTACION,DESARROLLO
2,3,130518,AEA.NQ.RCo-1023,2638.00,87.00,GAS PLUS,EXPLOTACION,DESARROLLO
3,4,133429,AEA.NQ.Gu-1190,2758.00,None,GAS PLUS,EXPLOTACION,DESARROLLO
4,5,143846,APA.RN.EFO-144(d),3949.00,None,GAS PLUS,EXPLOTACION,DESARROLLO
5,6,146139,APA.RN.EFO-0141(d),3879.00,0.00,GAS PLUS,EXPLOTACION,DESARROLLO
6,7,157213,YEA.RN.EFO-303(d),3755.00,0.00,Sin Proyecto,EXPLOTACION,DESARROLLO
7,8,130519,AEA.NQ.RCo-1025,2700.00,None,GAS PLUS,EXPLOTACION,DESARROLLO
8,9,132356,AEA.NQ.ACO-7,3330.50,None,GAS PLUS,EXPLOTACION,DESARROLLO
9,10,130158,AEA.NQ.RCo-1026(I)d,2700.00,52.00,GAS PLUS,EXPLOTACION,DESARROLLO


In [75]:
# Duplicates after loading

result = execute_query(
    """
    SELECT COUNT(*) - COUNT(DISTINCT idpozo)
    FROM dim_well
    """
)

duplicates = result[0][0]

print(f"Duplicated wells: {duplicates}")

Duplicated wells: 0


## 4.3 Load dim_geology

### Objective

The objective of this dimension is to store the geological attributes associated with the wells, allowing the analysis of production and fracturing according to the characteristics of the reservoir and geological formation.

### Source Data
#### Source dataset
```python
production_df
fracture_df
```
#### Source column

|Dataset|Columns|
|-------|-------|
|production_df|formacion, subtipo_recurso|
|fracture_df|tipo_reservorio, subtipo_reservorio, formacion_productiva|

The dim_geology dimension combines geological attributes from both source datasets.
The production dataset provides the general geological characteristics associated with each well, including the complete geological formation name and the hydrocarbon resource subtype.
The fracturing dataset provides additional reservoir information generated during hydraulic fracturing operations, including the reservoir classification, reservoir subtype, and productive formation identifier.
These attributes are combined using the well identifier (idpozo) to create a unified geological profile for each well. The complete formation name from production_df is preserved as the main geological reference, while the fracturing attributes are incorporated as additional analytical descriptors.

### Transformation

The following transformations will be applied:

1. Extract the geological attributes required from both source datasets.
2. Combine production and fracturing information using the well identifier (idpozo).
3. Preserve the complete geological formation name from production_df.
4. Rename formprod from fracture_df as formacion_productiva to match the dimensional model definition.
5. Remove duplicate geological profiles.
6. Generate the final DataFrame following the dim_geology table structure.

### Load

Insert the transformed records into the dim_geology table.
The surrogate key (geology_key) will be automatically generated by MySQL using the AUTO_INCREMENT property.

### Validation

After loading the data:

- Verify the total number of inserted records.
- Inspect the table structure.
- Verify that duplicate geological profiles were not generated.
- Confirm that the surrogate key was generated correctly.
- Inspect a sample of loaded records.

### Implementation

In [76]:
# =======================================================================
# 1. Extract
# =======================================================================

production_df = production_df.rename(columns={
    "sub_tipo_recurso": "subtipo_recurso"
})

production_geo = production_df[
    [
        "idpozo",
        "formacion", 
        "subtipo_recurso"
    ]
].copy()

fracture_geo = fracture_df[
    [
        "idpozo",
        "tipo_reservorio", 
        "subtipo_reservorio", 
        "formacion_productiva"
    ]
].copy()

production_geo.head()

,idpozo,formacion,subtipo_recurso
0,132772,basamento,TIGHT
1,130300,precuyo,TIGHT
2,130518,precuyo,TIGHT
3,133429,precuyo,TIGHT
4,143846,lajas,TIGHT


In [77]:
fracture_geo.head()

,idpozo,tipo_reservorio,subtipo_reservorio,formacion_productiva
0,159910,NO CONVENCIONAL,SHALE,los molles
1,159910,NO CONVENCIONAL,SHALE,los molles
2,159219,NO CONVENCIONAL,SHALE,vaca muerta
3,159220,NO CONVENCIONAL,SHALE,vaca muerta
4,159221,NO CONVENCIONAL,SHALE,vaca muerta


In [ ]:
# =======================================================================
# 2. Transform
# =======================================================================

production_geo = (
    production_geo
    .drop_duplicates(subset="idpozo")
    .reset_index(drop=True)
)

fracture_geo = (
    fracture_geo
    .drop_duplicates(subset="idpozo")
    .reset_index(drop=True)
)

In [80]:
print(production_geo.shape)
print(fracture_geo.shape)

(5041, 3)
(4562, 4)


In [81]:
# Merge production and fracturing information

dim_geo_df = production_geo.merge(
    fracture_geo,
    on="idpozo",
    how="left"
)

In [82]:
dim_geo_df.head()

,idpozo,formacion,subtipo_recurso,tipo_reservorio,subtipo_reservorio,formacion_productiva
0,132772,basamento,TIGHT,NaN,NaN,NaN
1,130300,precuyo,TIGHT,NaN,NaN,NaN
2,130518,precuyo,TIGHT,NO CONVENCIONAL,TIGHT,precuyo
3,133429,precuyo,TIGHT,NaN,NaN,NaN
4,143846,lajas,TIGHT,NaN,NaN,NaN


In [83]:
dim_geo_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5041 entries, 0 to 5040
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   idpozo                5041 non-null   int64 
 1   formacion             5041 non-null   object
 2   subtipo_recurso       5037 non-null   object
 3   tipo_reservorio       3765 non-null   object
 4   subtipo_reservorio    3761 non-null   object
 5   formacion_productiva  3765 non-null   object
dtypes: int64(1), object(5)
memory usage: 236.4+ KB


In [84]:
print(dim_geo_df.shape)
print(dim_geo_df["idpozo"].duplicated().sum())
print(dim_geo_df["idpozo"].nunique())

(5041, 6)
0
5041


In [85]:
dim_geo_df = dim_geo_df.drop(
    columns=["idpozo"]
).drop_duplicates()

In [86]:
dim_geo_df.head(10)

,formacion,subtipo_recurso,tipo_reservorio,subtipo_reservorio,formacion_productiva
0,basamento,TIGHT,NaN,NaN,NaN
1,precuyo,TIGHT,NaN,NaN,NaN
2,precuyo,TIGHT,NO CONVENCIONAL,TIGHT,precuyo
4,lajas,TIGHT,NaN,NaN,NaN
5,lajas,TIGHT,NO CONVENCIONAL,TIGHT,lajas
24,quintuco,TIGHT,NaN,NaN,NaN
42,formación improductiva,TIGHT,NaN,NaN,NaN
48,vaca muerta,SHALE,NaN,NaN,NaN
73,formación improductiva,TIGHT,NO CONVENCIONAL,TIGHT,formación improductiva
186,lotena,TIGHT,NaN,NaN,NaN


In [87]:
print(dim_geo_df.shape)

(58, 5)


In [88]:
dim_geo_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 58 entries, 0 to 4896
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   formacion             58 non-null     object
 1   subtipo_recurso       55 non-null     object
 2   tipo_reservorio       21 non-null     object
 3   subtipo_reservorio    18 non-null     object
 4   formacion_productiva  21 non-null     object
dtypes: object(5)
memory usage: 2.7+ KB


In [89]:
print(f"Unique geological profiles: {len(dim_geo_df)}")

Unique geological profiles: 58


In [90]:
print(
    "Duplicated geological profiles:",
    dim_geo_df.duplicated().sum()
)

Duplicated geological profiles: 0


In [91]:
# =======================================================================
# 3. Load
# =======================================================================

insert_sql = """
INSERT INTO dim_geology (
    formacion,
    subtipo_recurso,
    tipo_reservorio,
    subtipo_reservorio,
    formacion_productiva
)
VALUES (%s, %s, %s, %s, %s)
"""

load_to_table(insert_sql, dim_geo_df, "dim_geology")

✓ 58 records inserted into 'dim_geology'.


In [92]:
# =======================================================================
# 4. Validation
# =======================================================================

# Verify total number of records

print(f"Rows in dim_geology: {get_row_count('dim_geology')}")

Rows in dim_geology: 116


In [93]:
# Verify table structure

describe_table("dim_geology")

,Field,Type,Null,Key,Default,Extra
0,geology_key,int,NO,PRI,None,auto_increment
1,formacion,varchar(100),YES,,None,
2,subtipo_recurso,varchar(50),YES,,None,
3,tipo_reservorio,varchar(50),YES,,None,
4,subtipo_reservorio,varchar(50),YES,,None,
5,formacion_productiva,varchar(100),YES,,None,


In [94]:
# Verify SQL definition

print(show_create_table("dim_geology"))

CREATE TABLE `dim_geology` (
  `geology_key` int NOT NULL AUTO_INCREMENT,
  `formacion` varchar(100) COLLATE utf8mb4_unicode_ci DEFAULT NULL,
  `subtipo_recurso` varchar(50) COLLATE utf8mb4_unicode_ci DEFAULT NULL,
  `tipo_reservorio` varchar(50) COLLATE utf8mb4_unicode_ci DEFAULT NULL,
  `subtipo_reservorio` varchar(50) COLLATE utf8mb4_unicode_ci DEFAULT NULL,
  `formacion_productiva` varchar(100) COLLATE utf8mb4_unicode_ci DEFAULT NULL,
  PRIMARY KEY (`geology_key`)
) ENGINE=InnoDB AUTO_INCREMENT=117 DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci


In [95]:
# Inspect loaded records

pd.DataFrame(
    execute_query(
        """
        SELECT *
        FROM dim_geology
        LIMIT 10
        """
    ),
    columns=[
        "geology_key",
        "formacion",
        "subtipo_recurso",
        "tipo_reservorio",
        "subtipo_reservorio",
        "formacion_productiva"
    ]
)

,geology_key,formacion,subtipo_recurso,tipo_reservorio,subtipo_reservorio,formacion_productiva
0,1,basamento,TIGHT,None,None,None
1,2,precuyo,TIGHT,None,None,None
2,3,precuyo,TIGHT,NO CONVENCIONAL,TIGHT,precuyo
3,4,lajas,TIGHT,None,None,None
4,5,lajas,TIGHT,NO CONVENCIONAL,TIGHT,lajas
5,6,quintuco,TIGHT,None,None,None
6,7,formación improductiva,TIGHT,None,None,None
7,8,vaca muerta,SHALE,None,None,None
8,9,formación improductiva,TIGHT,NO CONVENCIONAL,TIGHT,formación improductiva
9,10,lotena,TIGHT,None,None,None


In [96]:
# Duplicates after loading

result = execute_query(
    """
    SELECT COUNT(*) - COUNT(DISTINCT CONCAT(
        COALESCE(formacion,''),
        COALESCE(subtipo_recurso,''),
        COALESCE(tipo_reservorio,''),
        COALESCE(subtipo_reservorio,''),
        COALESCE(formacion_productiva,'')
    ))
    FROM dim_geology
    """ 
)

duplicates = result[0][0]

print(f"Duplicated geo: {duplicates}")

Duplicated geo: 58


## 4.4 Load dim_field_area

### Objective

Populate the dim_field_area table with the unique field and operational area combinations extracted from the production dataset.

This dimension centralizes the geographical production hierarchy associated with each well and allows both fact tables to reference a common business location without storing descriptive text repeatedly.

### Source Data
#### Source dataset
```python
production_df
fracture_df
```

#### Source column

|Dataset|Columns|
|-------|-------|
|production_df|idareapermisoconcesion, areapermisoconcesion, idareayacimiento, areayacimiento|
|fracture_df|yacimiento|

The concession area identifiers and names (idareapermisoconcesion, areapermisoconcesion, idareayacimiento, and areayacimiento) are obtained from the production dataset, while the producing field (yacimiento) is obtained from the fracturing dataset.

These attributes are combined to build a single dimension representing the operational concession area and producing field hierarchy. This dimension will be referenced by both fact tables through a surrogate key (field_area_key), eliminating duplicated descriptive information and ensuring consistency across analytical queries.

### Transformation

The following transformations will be applied:

1. Extract the concession area attributes from the production dataset:
- idareapermisoconcesion
- areapermisoconcesion
- idareayacimiento
- areayacimiento
2. Extract the producing field (yacimiento) from the fracturing dataset.
3. Merge both datasets using idpozo as the common business key.
4. Remove the idpozo column after the merge since it belongs to the well dimension.
5. Remove duplicated records to obtain a unique set of concession area and producing field profiles.
6. Build the final DataFrame following the dimensional model definition.

### Load

Insert the transformed records into the dim_field_area table.

The surrogate key (field_area_key) will be automatically generated by MySQL using the AUTO_INCREMENT property.

### Validation

After loading the data:

- Verify the total number of inserted records.
- Inspect the table structure.
- Verify that duplicate field-area profiles were not generated.
- Verify that the surrogate key was generated correctly.
- Inspect a sample of loaded records.

### Implementation

In [108]:
# =======================================================================
# 1. Extract
# =======================================================================

production_area = production_df[
    [
        "idpozo",
        "idareapermisoconcesion",
        "areapermisoconcesion",
        "idareayacimiento",
        "areayacimiento"
    ]
].copy()

fracture_area = fracture_df[
    [
        "idpozo",
        "yacimiento"
    ]
].copy()

production_area.head()

,idpozo,idareapermisoconcesion,areapermisoconcesion,idareayacimiento,areayacimiento
0,132772,NDD,AL NORTE DE LA DORSAL,GUA,GUANACO
1,130300,SDD,AL SUR DE LA DORSAL,RQC,RANQUIL CO
2,130518,SDD,AL SUR DE LA DORSAL,RQC,RANQUIL CO
3,133429,NDD,AL NORTE DE LA DORSAL,GUA,GUANACO
4,143846,FEO,ESTACION FERNANDEZ ORO,Z155,ESTACION FERNANDEZ ORO


In [109]:
fracture_area.head()

,idpozo,yacimiento
0,159910,AGUA DEL CAJON
1,159910,AGUA DEL CAJON
2,159219,AGUADA DE LA ARENA
3,159220,AGUADA DE LA ARENA
4,159221,AGUADA DE LA ARENA


In [110]:
# =======================================================================
# 2. Transform
# =======================================================================

production_area = (
    production_area
    .drop_duplicates(subset="idpozo")
    .reset_index(drop=True)
)

fracture_area = (
    fracture_area
    .drop_duplicates(subset="idpozo")
    .reset_index(drop=True)
)

In [111]:
print(production_area.shape)
print(fracture_area.shape)

(5041, 5)
(4562, 2)


In [112]:
# Merge production and fracturing information

dim_field_area_df = production_area.merge(
    fracture_area,
    on="idpozo",
    how="left"
)

In [120]:
dim_field_area_df.head()

,idpozo,idareapermisoconcesion,areapermisoconcesion,idareayacimiento,areayacimiento,yacimiento
0,132772,NDD,AL NORTE DE LA DORSAL,GUA,GUANACO,NaN
1,130300,SDD,AL SUR DE LA DORSAL,RQC,RANQUIL CO,NaN
2,130518,SDD,AL SUR DE LA DORSAL,RQC,RANQUIL CO,RANQUIL CO
3,133429,NDD,AL NORTE DE LA DORSAL,GUA,GUANACO,NaN
4,143846,FEO,ESTACION FERNANDEZ ORO,Z155,ESTACION FERNANDEZ ORO,NaN


In [118]:
dim_field_area_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5041 entries, 0 to 5040
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   idpozo                  5041 non-null   int64 
 1   idareapermisoconcesion  5041 non-null   object
 2   areapermisoconcesion    5041 non-null   object
 3   idareayacimiento        5041 non-null   object
 4   areayacimiento          5041 non-null   object
 5   yacimiento              3765 non-null   object
dtypes: int64(1), object(5)
memory usage: 236.4+ KB


In [119]:
print(dim_field_area_df.shape)
print(dim_field_area_df["idpozo"].duplicated().sum())
print(dim_field_area_df["idpozo"].nunique())

(5041, 6)
0
5041


In [121]:
dim_field_area_df = dim_field_area_df.drop(
    columns=["idpozo"]
).drop_duplicates()

In [122]:
dim_field_area_df.head(10)

,idareapermisoconcesion,areapermisoconcesion,idareayacimiento,areayacimiento,yacimiento
0,NDD,AL NORTE DE LA DORSAL,GUA,GUANACO,NaN
1,SDD,AL SUR DE LA DORSAL,RQC,RANQUIL CO,NaN
2,SDD,AL SUR DE LA DORSAL,RQC,RANQUIL CO,RANQUIL CO
4,FEO,ESTACION FERNANDEZ ORO,Z155,ESTACION FERNANDEZ ORO,NaN
5,FEO,ESTACION FERNANDEZ ORO,Z155,ESTACION FERNANDEZ ORO,ESTACION FERNANDEZ ORO
8,ANC,ANTICLINAL CAMPAMENTO,ACO,ANTICLINAL CAMPAMENTO OESTE,NaN
11,ANC,ANTICLINAL CAMPAMENTO,ANCS,ANTICLINAL CAMPAMENTO SUR,NaN
48,X008,CORTADERA,Y310,CORTADERA,NaN
125,X009,HUACALERA,Y325,HUACALERA,NaN
126,ANC,ANTICLINAL CAMPAMENTO,ANC,ANTICLINAL CAMPAMENTO,NaN


In [123]:
dim_field_area_df.shape

(234, 5)

In [124]:
dim_field_area_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 234 entries, 0 to 5040
Data columns (total 5 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   idareapermisoconcesion  234 non-null    object
 1   areapermisoconcesion    234 non-null    object
 2   idareayacimiento        234 non-null    object
 3   areayacimiento          234 non-null    object
 4   yacimiento              103 non-null    object
dtypes: object(5)
memory usage: 11.0+ KB


In [125]:
print(f"Unique Field Area profiles: {len(dim_field_area_df)}")

Unique Field Area profiles: 234


In [127]:
print(
    "Duplicated field area profiles:",
    dim_field_area_df.duplicated().sum()
)

Duplicated field area profiles: 0


In [128]:
print(
    production_df["areayacimiento"]
    .sort_values()
    .unique()[:20]
)

print(
    fracture_df["yacimiento"]
    .sort_values()
    .unique()[:20]
)

['AGUA DE AFUERA' 'AGUA DEL CAJON' 'AGUADA CÁNEPA' 'AGUADA DE CASTRO'
 'AGUADA DE LA ARENA' 'AGUADA DEL CHIVATO' 'AGUADA DEL PONCHO'
 'AGUADA FEDERAL' 'AGUADA LOS LOROS' 'AGUADA PICHANA ESTE MULICHINCO'
 'AGUADA PICHANA ESTE VACA MUERTA' 'AGUADA PICHANA OESTE'
 'AGUADA SAN ROQUE' 'AGUADA VILLANUEVA NORTE' 'AGUILA MORA' 'ANGOSTURA'
 'ANTICLINAL CAMPAMENTO' 'ANTICLINAL CAMPAMENTO OESTE'
 'ANTICLINAL CAMPAMENTO SUR' 'BAJADA DE AÑELO']
['AGUA DE AFUERA' 'AGUA DEL CAJON' 'AGUADA CÁNEPA' 'AGUADA DE CASTRO'
 'AGUADA DE LA ARENA' 'AGUADA DE LOS INDIOS' 'AGUADA DE LOS INDIOS SUR'
 'AGUADA DEL CHIVATO' 'AGUADA FEDERAL' 'AGUADA LOS LOROS'
 'AGUADA PICHANA ESTE MULICHINCO' 'AGUADA PICHANA ESTE VACA MUERTA'
 'AGUADA PICHANA OESTE' 'AGUADA VILLANUEVA NORTE' 'AGUILA MORA'
 'BAJADA DE AÑELO' 'BAJADA DEL PALO ESTE' 'BAJADA DEL PALO OESTE'
 'BAJO DEL CHOIQUE' 'BAJO DEL TORO']


In [131]:
# =======================================================================
# 3. Load
# =======================================================================

insert_sql = """
INSERT INTO dim_field_Area (
    idareapermisoconcesion,
    areapermisoconcesion,
    idareayacimiento,
    areayacimiento,
    yacimiento 
)
VALUES (%s, %s, %s, %s, %s)
"""

load_to_table(insert_sql, dim_field_area_df, "dim_field_area")

✓ 234 records inserted into 'dim_field_area'.


In [132]:
# =======================================================================
# 4. Validation
# =======================================================================

# Verify total number of records

print(f"Rows in dim_field_area: {get_row_count('dim_field_area')}")

Rows in dim_field_area: 234


In [133]:
# Verify table structure

describe_table("dim_field_area")

,Field,Type,Null,Key,Default,Extra
0,field_area_key,int,NO,PRI,None,auto_increment
1,idareapermisoconcesion,varchar(20),YES,,None,
2,areapermisoconcesion,varchar(150),YES,,None,
3,idareayacimiento,varchar(20),YES,,None,
4,areayacimiento,varchar(150),YES,,None,
5,yacimiento,varchar(150),YES,,None,


In [134]:
# Verify SQL definition

print(show_create_table("dim_field_area"))

CREATE TABLE `dim_field_area` (
  `field_area_key` int NOT NULL AUTO_INCREMENT,
  `idareapermisoconcesion` varchar(20) COLLATE utf8mb4_unicode_ci DEFAULT NULL,
  `areapermisoconcesion` varchar(150) COLLATE utf8mb4_unicode_ci DEFAULT NULL,
  `idareayacimiento` varchar(20) COLLATE utf8mb4_unicode_ci DEFAULT NULL,
  `areayacimiento` varchar(150) COLLATE utf8mb4_unicode_ci DEFAULT NULL,
  `yacimiento` varchar(150) COLLATE utf8mb4_unicode_ci DEFAULT NULL,
  PRIMARY KEY (`field_area_key`)
) ENGINE=InnoDB AUTO_INCREMENT=235 DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci


In [135]:
# Inspect loaded records

pd.DataFrame(
    execute_query(
        """
        SELECT *
        FROM dim_field_area
        LIMIT 10
        """
    ),
    columns=[
        "field_area_key",
        "idareapermisoconcesion",
        "areapermisoconcesion",
        "idareayacimiento",
        "areayacimiento",
        "yacimiento",
    ]
)

,field_area_key,idareapermisoconcesion,areapermisoconcesion,idareayacimiento,areayacimiento,yacimiento
0,1,NDD,AL NORTE DE LA DORSAL,GUA,GUANACO,None
1,2,SDD,AL SUR DE LA DORSAL,RQC,RANQUIL CO,None
2,3,SDD,AL SUR DE LA DORSAL,RQC,RANQUIL CO,RANQUIL CO
3,4,FEO,ESTACION FERNANDEZ ORO,Z155,ESTACION FERNANDEZ ORO,None
4,5,FEO,ESTACION FERNANDEZ ORO,Z155,ESTACION FERNANDEZ ORO,ESTACION FERNANDEZ ORO
5,6,ANC,ANTICLINAL CAMPAMENTO,ACO,ANTICLINAL CAMPAMENTO OESTE,None
6,7,ANC,ANTICLINAL CAMPAMENTO,ANCS,ANTICLINAL CAMPAMENTO SUR,None
7,8,X008,CORTADERA,Y310,CORTADERA,None
8,9,X009,HUACALERA,Y325,HUACALERA,None
9,10,ANC,ANTICLINAL CAMPAMENTO,ANC,ANTICLINAL CAMPAMENTO,None


In [138]:
# Duplicates after loading

duplicates = execute_query("""
SELECT COUNT(*) - COUNT(DISTINCT CONCAT(
    COALESCE(idareapermisoconcesion,''),
    COALESCE(areapermisoconcesion,''),
    COALESCE(idareayacimiento,''),
    COALESCE(areayacimiento,''),
    COALESCE(yacimiento,'')
))
FROM dim_field_area
""")[0][0]

print(f"Duplicated field area profiles: {duplicates}")

Duplicated field area profiles: 0


In [139]:
result = execute_query("""
SELECT COUNT(*)
FROM dim_field_area
WHERE yacimiento IS NULL
""")

missing = result[0][0]

print(f"Profiles without fracturing field: {missing}")

Profiles without fracturing field: 131


In [141]:
result = execute_query("""
SELECT
    COUNT(*) AS total_profiles,
    COUNT(yacimiento) AS with_fracturing_field,
    COUNT(*) - COUNT(yacimiento) AS without_fracturing_field
FROM dim_field_area
""")

pd.DataFrame(
    result,
    columns=[
        "total_profiles",
        "with_fracturing_field",
        "without_fracturing_field"
    ]
)

,total_profiles,with_fracturing_field,without_fracturing_field
0,234,103,131


## 4.5 Load dim_location

### Objective

Populate the dim_location table with the unique geographical locations extracted from the production dataset.

This dimension centralizes the geographical information associated with each well, allowing both fact tables to reference a standardized location through a surrogate key while avoiding redundant storage of geographical attributes.

### Source Data

#### Source dataset
```python
production_df
```

#### Source columns

|Dataset|Columns|
|-------|-------|
|production_df|cuenca, provincia, coordenadax, coordenaday|

The production dataset contains the geographical attributes that identify the location of each well. These include the sedimentary basin (cuenca), the Argentine province (provincia), and the geographic coordinates (coordenadax and coordenaday).

These attributes will be consolidated into a single dimension containing unique geographical location profiles.

#### Transformation

The following transformations will be applied:

1. Extract the geographical attributes:
- cuenca
- provincia
- coordenadax
- coordenaday
2. Remove duplicated geographical profiles.
3. Verify that each location profile appears only once.
4. Build the final DataFrame following the dimensional model definition.

#### Load

Insert the transformed records into the dim_location table.

The surrogate key (location_key) will be automatically generated by MySQL using the AUTO_INCREMENT property.

#### Validation

After loading the data:

- Verify the total number of inserted records.
- Inspect the table structure.
- Verify that duplicate geographical profiles were not generated.
- Verify that the surrogate key was generated correctly.
- Inspect a sample of loaded records.

### Implementation

In [11]:
# =======================================================================
# 1. Extract
# =======================================================================

production_location = production_df[
    [
        "idpozo",
        "cuenca",
        "provincia",
        "coordenadax",
        "coordenaday"
    ]
].copy()

production_location.head()

,idpozo,cuenca,provincia,coordenadax,coordenaday
0,132772,NEUQUINA,Neuquén,-69.232012,-38.862477
1,130300,NEUQUINA,Neuquén,-69.193650,-39.085770
2,130518,NEUQUINA,Neuquén,-69.201125,-39.085068
3,133429,NEUQUINA,Neuquén,-69.224719,-38.867785
4,143846,NEUQUINA,Rio Negro,-67.834228,-39.030246


In [12]:
# =======================================================================
# 2. Transform
# =======================================================================

dim_location_df = (
    production_location
    .drop(columns=["idpozo"])
    .copy()
)

In [14]:
# Match MySQL DECIMAL(12,8) precision

dim_location_df["coordenadax"] = (
    dim_location_df["coordenadax"]
    .round(8)
)

In [15]:
dim_location_df["coordenaday"] = (
    dim_location_df["coordenaday"]
    .round(8)
)

In [16]:
# Remove duplicated location profiles

dim_location_df = (
    dim_location_df
    .drop_duplicates()
    .reset_index(drop=True)
)

In [17]:
print(dim_location_df.shape)

(4841, 4)


In [18]:
print(
    "Duplicated location profiles:",
    dim_location_df.duplicated().sum()
)

Duplicated location profiles: 0


#### Observation

Geographic coordinates were rounded to 8 decimal places to match the target database precision defined as DECIMAL(12,8). After precision alignment, duplicated location profiles were removed based on the complete geographical attribute combination.

In [21]:
# =======================================================================
# 3. Load
# =======================================================================

insert_sql = """
INSERT INTO dim_location (
    cuenca,
    provincia,
    coordenadax,
    coordenaday 
)
VALUES (%s, %s, %s, %s)
"""

load_to_table(insert_sql, dim_location_df, "dim_location")

✓ 4841 records inserted into 'dim_location'.


In [22]:
# =======================================================================
# 4. Validation
# =======================================================================

# Verify total number of records

print(f"Rows in dim_location: {get_row_count('dim_location')}")

Rows in dim_location: 4841


In [23]:
# Verify table structure

describe_table("dim_location")

,Field,Type,Null,Key,Default,Extra
0,location_key,int,NO,PRI,None,auto_increment
1,cuenca,varchar(100),YES,,None,
2,provincia,varchar(100),YES,,None,
3,coordenadax,"decimal(12,8)",YES,,None,
4,coordenaday,"decimal(12,8)",YES,,None,


In [24]:
# Verify SQL definition

print(show_create_table("dim_location"))

CREATE TABLE `dim_location` (
  `location_key` int NOT NULL AUTO_INCREMENT,
  `cuenca` varchar(100) COLLATE utf8mb4_unicode_ci DEFAULT NULL,
  `provincia` varchar(100) COLLATE utf8mb4_unicode_ci DEFAULT NULL,
  `coordenadax` decimal(12,8) DEFAULT NULL,
  `coordenaday` decimal(12,8) DEFAULT NULL,
  PRIMARY KEY (`location_key`)
) ENGINE=InnoDB AUTO_INCREMENT=4842 DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci


In [25]:
# Inspect loaded records

pd.DataFrame(
    execute_query(
        """
        SELECT *
        FROM dim_location
        LIMIT 10
        """
    ),
    columns=[
        "location_key",
        "cuenca",
        "provincia",
        "coordenadax",
        "coordenaday"
    ]
)

,location_key,cuenca,provincia,coordenadax,coordenaday
0,1,NEUQUINA,Neuquén,-69.23201187,-38.86247715
1,2,NEUQUINA,Neuquén,-69.19364995,-39.08576987
2,3,NEUQUINA,Neuquén,-69.20112501,-39.08506840
3,4,NEUQUINA,Neuquén,-69.22471911,-38.86778528
4,5,NEUQUINA,Rio Negro,-67.83422842,-39.03024615
5,6,NEUQUINA,Rio Negro,-67.82324600,-39.02327200
6,7,NEUQUINA,Rio Negro,-67.82391846,-39.01171050
7,8,NEUQUINA,Neuquén,-69.19000214,-39.07992860
8,9,NEUQUINA,Neuquén,-69.79236159,-38.95381048
9,10,NEUQUINA,Neuquén,-69.19380184,-39.08096074


In [26]:
# Duplicates after loading

duplicates = execute_query("""
SELECT
    cuenca,
    provincia,
    coordenadax,
    coordenaday,
    COUNT(*) AS cantidad
FROM dim_location
GROUP BY
    cuenca,
    provincia,
    coordenadax,
    coordenaday
HAVING COUNT(*) > 1;
""")

In [27]:
pd.DataFrame(
    duplicates,
    columns=[
        "cuenca",
        "provincia",
        "coordenadax",
        "coordenaday",
        "cantidad"
    ]
)

,cuenca,provincia,coordenadax,coordenaday,cantidad


In [29]:
print("Python:", len(dim_location_df))
print("MySQL:", get_row_count("dim_location"))

Python: 4841
MySQL: 4841


- Verify the number of inserted records.
- Verify that the surrogate key was generated automatically.
- Inspect the table structure.
- Compare Python transformed records against MySQL loaded records.
- Verify that no duplicated geographic profiles exist.

## 4.5 Load dim_company

### Objective

Populate the dim_company table with the unique operating companies identified in the production dataset.

This dimension centralizes company information and provides a standardized reference for analytical reporting. The surrogate key generated by this dimension will later be used by the fact tables to relate production and operational records with their corresponding operating company.

### Source Data

#### Source dataset

```python
production_df
```

#### Source column

|Dataset|Columns|
|-------|-------|
|production_df|idempresa, empresa|

The production dataset contains the operating company information associated with each well production record.

The idempresa column represents the natural identifier assigned by the source system, while empresa contains the official company name.

These attributes will be extracted and transformed into a dedicated company dimension, avoiding redundant storage of company information in fact tables.

### Transformation

The following transformations will be applied:

1. Extract the company attributes:
- idempresa
- empresa
2. Remove duplicated company records.
3. Verify that each natural company identifier (idempresa) represents a unique company.
4. Build the final DataFrame following the dimensional model definition.

### Load

Insert the transformed records into the dim_company table.

The surrogate key (company_key) will be automatically generated by MySQL using the AUTO_INCREMENT property.

### Validation

After loading the data:

- Verify the total number of inserted records.
- Inspect the table structure.
- Verify that one record exists per company natural identifier (idempresa).
- Verify that the surrogate key was generated correctly.
- Inspect a sample of loaded records.

### Implementation

In [12]:
# =======================================================================
# 1. Extract
# =======================================================================

production_company = production_df[
    [
        "idempresa",
        "empresa"
    ]
].copy()

In [13]:
production_company.head()

,idempresa,empresa
0,YSUR,YSUR ENERGÍA ARGENTINA S.R.L.
1,YSUR,YSUR ENERGÍA ARGENTINA S.R.L.
2,YSUR,YSUR ENERGÍA ARGENTINA S.R.L.
3,YSUR,YSUR ENERGÍA ARGENTINA S.R.L.
4,YSUR,YSUR ENERGÍA ARGENTINA S.R.L.


In [14]:
production_company.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 415903 entries, 0 to 415902
Data columns (total 2 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   idempresa  415903 non-null  object
 1   empresa    415903 non-null  object
dtypes: object(2)
memory usage: 6.3+ MB


In [ ]:
# =======================================================================
# 2. Transformation
# =======================================================================

dim_company_df = (
    production_company
    .drop_duplicates(subset="idempresa")
    .reset_index(drop=True)
)

In [16]:
print(dim_company_df.shape)

print(
    dim_company_df["idempresa"].duplicated().sum()
)

print(
    dim_company_df["idempresa"].nunique()
)

(55, 2)
0
55


The transformed company dimension was validated before loading into the database.

Validation results:

- Total unique company records: 55
- Duplicated idempresa values: 0
- Unique idempresa values: 55

This confirms that:

- Each operating company appears only once in the dimension.
- The natural key (idempresa) uniquely identifies each company.
- The DataFrame is ready to be loaded into the dim_company table

In [17]:
# =======================================================================
# 3. Load
# =======================================================================

insert_sql = """
INSERT INTO dim_company (
    idempresa,
    empresa
)
VALUES (%s, %s)
"""

load_to_table(insert_sql, dim_company_df, "dim_company")

✓ 55 records inserted into 'dim_company'.


In [18]:
# =======================================================================
# 4. Validation
# =======================================================================

# Verify total number of records
print(f"Rows in dim_company: {get_row_count('dim_company')}")

Rows in dim_company: 55


In [19]:
# Verify table structure

describe_table("dim_company")

,Field,Type,Null,Key,Default,Extra
0,company_key,int,NO,PRI,None,auto_increment
1,idempresa,varchar(20),NO,UNI,None,
2,empresa,varchar(150),NO,,None,


In [20]:
# Verify SQL definition

print(show_create_table("dim_company"))

CREATE TABLE `dim_company` (
  `company_key` int NOT NULL AUTO_INCREMENT,
  `idempresa` varchar(20) COLLATE utf8mb4_unicode_ci NOT NULL,
  `empresa` varchar(150) COLLATE utf8mb4_unicode_ci NOT NULL,
  PRIMARY KEY (`company_key`),
  UNIQUE KEY `idempresa` (`idempresa`)
) ENGINE=InnoDB AUTO_INCREMENT=56 DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci


In [21]:
# Inspect loaded records

pd.DataFrame(
    execute_query(
        """
        SELECT *
        FROM dim_company
        LIMIT 10
        """
    ),
    columns=[
        "company_key",
        "idempresa",
        "empresa"
    ]
)

,company_key,idempresa,empresa
0,1,YSUR,YSUR ENERGÍA ARGENTINA S.R.L.
1,2,YPF,YPF S.A.
2,3,WIN,WINTERSHALL ENERGIA S.A.
3,4,WDA,WINTERSHALL DEA ARGENTINA S.A
4,5,VST,VISTA ENERGY ARGENTINA SAU
5,6,VOG,Vista Oil & Gas Argentina SA
6,7,VNO,VENOIL S.A.
7,8,VMI,VACA MUERTA INVERSIONES SAU
8,9,VIS,VISTA OIL & GAS ARGENTINA SAU
9,10,TPT,TECPETROL S.A.


In [23]:
# Duplicates after loading

duplicates = execute_query("""
SELECT COUNT(*) - COUNT(DISTINCT idempresa)
FROM dim_company
""")[0][0]

print(f"Duplicated company IDs: {duplicates}")

Duplicated company IDs: 0
